# Bank Marketing Response Prediction

Explore the data, compare candidate models on the chronological validation segment, and perform one final evaluation on the last chronological segment. The final segment has already been inspected and scored in earlier notebook versions, so its results are reported as **previously exposed**, not as an untouched estimate.


## 1. Environment and Dataset


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt

from bank_marketing.preprocessing import (
    EXCLUDED_FEATURES, chronological_split, load_dataset, select_features
)
from bank_marketing.evaluate import evaluate_probabilities
from bank_marketing.train import compare_validation_models
from bank_marketing.models import build_candidates
from bank_marketing.finalize import run_final_evaluation

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data_path = project_root / "data" / "raw" / "bank-additional-full.csv"

print("Python:", sys.version.split()[0])
print("Environment:", sys.executable)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)


Python: 3.13.5
Environment: D:\code\Bank Marketing Response Prediction\bank-marketing-ml\.venv\Scripts\python.exe
NumPy: 2.5.4
Pandas: 3.1.0
Scikit-learn: 1.9.1


In [2]:
df = load_dataset(data_path)
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())


Dataset shape: (41188, 21)
Columns: ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y']


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


## 2. Data Inspection


In [3]:
print("Data types:")
display(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nTarget distribution:")
print(df["y"].value_counts())

print("\nTarget percentage:")
print(df["y"].value_counts(normalize=True) * 100)

Data types:


age                 int64
job                   str
marital               str
education             str
default               str
housing               str
loan                  str
contact               str
month                 str
day_of_week           str
duration            int64
campaign            int64
pdays               int64
previous            int64
poutcome              str
emp.var.rate      float64
cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
y                     str
dtype: object


Missing values:
age               0
job               0
marital           0
education         0
default           0
housing           0
loan              0
contact           0
month             0
day_of_week       0
duration          0
campaign          0
pdays             0
previous          0
poutcome          0
emp.var.rate      0
cons.price.idx    0
cons.conf.idx     0
euribor3m         0
nr.employed       0
y                 0
dtype: int64

Target distribution:
y
no     36548
yes     4640
Name: count, dtype: int64

Target percentage:
y
no     88.734583
yes    11.265417
Name: proportion, dtype: float64


In [4]:
# 1. Check exact duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# 2. Count unknown values in each column
unknown_counts = (
    df.select_dtypes(include="object")
      .apply(lambda col: col.eq("unknown").sum())
)

print("\nUnknown values by column:")
print(unknown_counts[unknown_counts > 0].sort_values(ascending=False))

Duplicate rows: 12

Unknown values by column:
default      8597
education    1731
housing       990
loan          990
job           330
marital        80
dtype: int64


C:\Users\dhruv\AppData\Local\Temp\ipykernel_21772\945512961.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.select_dtypes(include="object")


In [5]:

# Inspect potentially problematic features

investigation_cols = [
    "duration",
    "campaign",
    "pdays",
    "previous",
    "poutcome"
]

print("Feature summaries:")
display(df[investigation_cols].describe(include="all").T)

print("\nTarget conversion rate by feature:")

for col in investigation_cols:
    print(f"\n--- {col} ---")
    display(
        df.groupby(col, dropna=False)["y"]
        .value_counts(normalize=True)
        .unstack(fill_value=0)
        .round(3)
        .head(15)
    )



Feature summaries:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
duration,41188.0,NaN,NaN,NaN,258.28501,259.279249,0.0,102.0,180.0,319.0,4918.0
campaign,41188.0,NaN,NaN,NaN,2.567593,2.770014,1.0,1.0,2.0,3.0,56.0
pdays,41188.0,NaN,NaN,NaN,962.475454,186.910907,0.0,999.0,999.0,999.0,999.0
previous,41188.0,NaN,NaN,NaN,0.172963,0.494901,0.0,0.0,0.0,0.0,7.0
poutcome,41188,3,nonexistent,35563,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Target conversion rate by feature:

--- duration ---


y,no,yes
duration,,
0,1.0,0.0
1,1.0,0.0
2,1.0,0.0
3,1.0,0.0
4,1.0,0.0
5,1.0,0.0
6,1.0,0.0
7,1.0,0.0
8,1.0,0.0



--- campaign ---


y,no,yes
campaign,,
1,0.870,0.130
2,0.885,0.115
3,0.893,0.107
4,0.906,0.094
5,0.925,0.075
6,0.923,0.077
7,0.940,0.060
8,0.958,0.042
9,0.940,0.060


... (2 outputs left out)


--- previous ---


y,no,yes
previous,,
0,0.912,0.088
1,0.788,0.212
2,0.536,0.464
3,0.407,0.593
4,0.457,0.543
5,0.278,0.722
6,0.400,0.600
7,1.000,0.000



--- poutcome ---


y,no,yes
poutcome,,
failure,0.858,0.142
nonexistent,0.912,0.088
success,0.349,0.651


In [6]:

# Check feature names, data types, and unknown values

feature_audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "unknown_count": (
        df.astype("string")
        .eq("unknown")
        .sum()
    ),
    "unique_values": df.nunique()
})

display(feature_audit)



,dtype,unknown_count,unique_values
age,int64,0,78
job,str,330,12
marital,str,80,4
education,str,1731,8
default,str,8597,3
housing,str,990,3
loan,str,990,3
contact,str,0,2
month,str,0,10
day_of_week,str,0,5


## 3. Prediction Timing and Feature Decisions


## 3. Prediction-Time Leakage Investigation

Investigate features that may contain information unavailable at the time of prediction.

Focus:
- duration
- campaign
- pdays
- previous
- poutcome

Goal: Decide which features are valid for our prediction scenario.




## 4. Feature Inclusion Decisions

### Objective
Decide which features are appropriate for predicting whether a client will subscribe to a term deposit.

### Feature Decisions

1. **duration — Exclude**
   Call duration is known only after the call ends. Using it for a pre-call prediction would cause target leakage.

2. **campaign — Review**
   Number of contacts during the current campaign. Its availability depends on when the prediction is made.

3. **pdays — Include with interpretation**
   Days since the client was last contacted in a previous campaign. The value 999 indicates that the client was not previously contacted.

4. **previous — Include**
   Number of contacts before the current campaign.

5. **poutcome — Include**
   Outcome of the previous marketing campaign.

### Prediction Setting
The model will predict subscription probability **before the current marketing call begins**.

Therefore, features must be available at that point in time.

### Important
Feature inclusion will be finalized after checking the prediction timing, data availability, and remaining feature definitions.


## 5. Final Feature Selection

### Prediction Objective
Predict whether a client will subscribe to a term deposit before the current call begins.

### Excluded Features
- duration: Known only after the call; target leakage.
- campaign: Includes the current contact, so exclude under our strict pre-call policy.

### Target
- y: Subscription outcome (yes/no).

### Selected Features
All remaining eligible predictors will be retained initially.
Their preprocessing will be handled separately.

### Policy
Only information available before the current call can be used.

In [7]:
X, y = select_features(df)
print("Target:", "y")
print("Excluded features:", list(EXCLUDED_FEATURES))
print("Selected predictors:", len(X.columns))
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("\nFull dataset target distribution:")
print(y.value_counts())


Target: y
Excluded features: ['duration', 'campaign']
Selected predictors: 18
Feature matrix shape: (41188, 18)
Target shape: (41188,)

Full dataset target distribution:
y
no     36548
yes     4640
Name: count, dtype: int64


## 4. Chronological Train, Validation, and Test Split

Keep the original row order. Model comparisons and threshold analysis use training and validation only. The final segment is not used until the final evaluation cell. Earlier notebook versions have already exposed it; that limitation is recorded in the final report.


In [8]:
splits = chronological_split(df)
X_fit, y_fit = splits.X_train, splits.y_train
X_val, y_val = splits.X_validation, splits.y_validation
X_test, y_test = splits.X_test, splits.y_test

print("Training rows:", len(X_fit))
print("Validation rows:", len(X_val))
print("Final test rows:", len(X_test))
print("Training index range:", X_fit.index.min(), "to", X_fit.index.max())
print("Validation index range:", X_val.index.min(), "to", X_val.index.max())
print("Final test index range:", X_test.index.min(), "to", X_test.index.max())
assert (len(X_fit), len(X_val), len(X_test)) == (26360, 6590, 8238)
print("\nTraining subscriber rate:", f"{(y_fit == 'yes').mean():.2%}")
print("Validation subscriber rate:", f"{(y_val == 'yes').mean():.2%}")


Training rows:

 26360
Validation rows: 6590
Final test rows: 8238
Training index range: 0 to 26359
Validation index range: 26360 to 32949
Final test index range: 32950 to 41187

Training subscriber rate: 4.79%
Validation subscriber rate: 12.72%


## 5. Chronological Distribution Shift

Compare the model-training and validation periods before candidate selection. This helps explain why a model can rank records moderately well while its probabilities and default-threshold predictions perform poorly.


In [9]:
train_monthly = pd.crosstab(X_fit["month"], y_fit, normalize="index") * 100
validation_monthly = pd.crosstab(X_val["month"], y_val, normalize="index") * 100
monthly_comparison = pd.DataFrame({
    "training_yes_%": train_monthly.get("yes", 0),
    "validation_yes_%": validation_monthly.get("yes", 0),
}).fillna(0)
display(monthly_comparison.round(2))
print("\nTraining month counts:")
print(X_fit["month"].value_counts())
print("\nValidation month counts:")
print(X_val["month"].value_counts())


,training_yes_%,validation_yes_%
month,,
apr,0.00,17.98
aug,5.24,0.00
dec,0.00,10.00
jul,6.09,0.00
jun,4.30,0.00
mar,0.00,44.68
may,3.09,7.66
nov,4.97,5.76
oct,62.69,0.00



Training month counts:
month
may    7763
jul    6685
aug    5175
jun    4374
nov    2296
oct      67
Name: count, dtype: int64

Validation month counts:
month
may    2520
apr    2458
nov    1320
mar     282
dec      10
Name: count, dtype: int64


## 6. Baseline Model

Fit the most-frequent DummyClassifier using the training segment and compare it on validation. The final test segment is not used here.


In [10]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_fit, y_fit)
y_val_prob_baseline = baseline.predict_proba(X_val)[:, list(baseline.classes_).index("yes")]
baseline_metrics = evaluate_probabilities(y_val, y_val_prob_baseline)
print(pd.Series({k: v for k, v in baseline_metrics.items() if k != "confusion_matrix"}).round(4))
print("Confusion matrix [no, yes]:")
print(np.asarray(baseline_metrics["confusion_matrix"]))


roc_auc                       0.5000
average_precision             0.1272
precision                     0.0000
recall                        0.0000
f1                            0.0000
accuracy                      0.8728
brier_score                   0.1272
actual_subscriber_rate        0.1272
mean_predicted_probability    0.0000
predicted_subscriber_rate     0.0000
threshold                     0.5000
dtype: float64
Confusion matrix [no, yes]:
[[5752    0]
 [ 838    0]]


## 7. Model Comparison and Limited Hyperparameter Checks

Compare Logistic Regression, class-weighted Logistic Regression, the existing Decision Tree, and a small set of Random Forest variants. A chronological sigmoid-calibrated forest is included as a diagnostic. Every preprocessor is fitted inside its model using training data only. Candidates are ranked by Average Precision, with ROC-AUC shown alongside it.


In [11]:
validation_results, validation_models, splits = compare_validation_models(df)
display(validation_results)
print("\nValidation subscriber rate (Average Precision reference):", f"{(y_val == 'yes').mean():.4f}")


,model,roc_auc,average_precision,precision,recall,f1,accuracy,brier_score,actual_subscriber_rate,mean_predicted_probability,predicted_subscriber_rate,threshold,confusion_matrix
0,random_forest_balanced,0.592581,0.157186,0.170006,0.334129,0.225352,0.707891,0.227683,0.127162,0.469864,0.249924,0.5,"[[4385, 1367], [558, 280]]"
1,random_forest_unweighted,0.599977,0.156845,0.000000,0.000000,0.000000,0.872838,0.116142,0.127162,0.052310,0.000000,0.5,"[[5752, 0], [838, 0]]"
2,random_forest_balanced_larger_leaf,0.587695,0.154964,0.166930,0.378282,0.231640,0.680880,0.233100,0.127162,0.477469,0.288164,0.5,"[[4170, 1582], [521, 317]]"
3,random_forest_balanced_shallower,0.584446,0.154682,0.161497,0.360382,0.223043,0.680728,0.230671,0.127162,0.473936,0.283763,0.5,"[[4184, 1568], [536, 302]]"
4,decision_tree,0.566171,0.144111,0.146726,0.866348,0.250951,0.342337,0.268123,0.127162,0.516180,0.750835,0.5,"[[1530, 4222], [112, 726]]"
5,logistic_regression,0.573090,0.135829,0.000000,0.000000,0.000000,0.872838,0.126599,0.127162,0.006660,0.000000,0.5,"[[5752, 0], [838, 0]]"
6,class_weighted_logistic,0.572913,0.135361,0.027027,0.002387,0.004386,0.862215,0.151048,0.127162,0.083582,0.011229,0.5,"[[5680, 72], [836, 2]]"
7,calibrated_random_forest,0.478086,0.118451,0.000000,0.000000,0.000000,0.872838,0.117013,0.127162,0.050010,0.000000,0.5,"[[5752, 0], [838, 0]]"



Validation subscriber rate (Average Precision reference): 0.1272


### Probability and Calibration Checks

Compare the mean predicted probability with the observed validation rate and inspect Brier score. Class weighting can shift predicted probabilities; the chronological calibration candidate is assessed on validation after calibration folds restricted to training. Calibration on this validation segment is not used.


In [12]:
probability_summary = validation_results[[
    "model", "average_precision", "brier_score",
    "actual_subscriber_rate", "mean_predicted_probability",
    "predicted_subscriber_rate"
]].copy()
display(probability_summary.round(4))


,model,average_precision,brier_score,actual_subscriber_rate,mean_predicted_probability,predicted_subscriber_rate
0,random_forest_balanced,0.1572,0.2277,0.1272,0.4699,0.2499
1,random_forest_unweighted,0.1568,0.1161,0.1272,0.0523,0.0000
2,random_forest_balanced_larger_leaf,0.1550,0.2331,0.1272,0.4775,0.2882
3,random_forest_balanced_shallower,0.1547,0.2307,0.1272,0.4739,0.2838
4,decision_tree,0.1441,0.2681,0.1272,0.5162,0.7508
5,logistic_regression,0.1358,0.1266,0.1272,0.0067,0.0000
6,class_weighted_logistic,0.1354,0.1510,0.1272,0.0836,0.0112
7,calibrated_random_forest,0.1185,0.1170,0.1272,0.0500,0.0000


## 8. Validation Threshold and Outreach Objective

The requested objective is to maximize recall subject to at least 0.50 precision. Check whether each candidate can meet that floor on validation. If no candidate meets it, do not claim the objective is satisfied.


In [13]:
from sklearn.metrics import precision_recall_curve

precision_floor = 0.50
threshold_rows = []
y_val_binary = (y_val == "yes").astype(int)
for name, fitted_model in validation_models.items():
    probabilities = fitted_model.predict_proba(X_val)[:, list(fitted_model.classes_).index("yes")]
    precision, recall, thresholds = precision_recall_curve(y_val_binary, probabilities)
    eligible = np.flatnonzero(precision[:-1] >= precision_floor)
    if len(eligible):
        best = eligible[np.argmax(recall[eligible])]
        threshold_rows.append({"model": name, "meets_precision_floor": True, "threshold": thresholds[best], "precision": precision[best], "recall": recall[best]})
    else:
        best = int(np.argmax(precision[:-1]))
        threshold_rows.append({"model": name, "meets_precision_floor": False, "threshold": thresholds[best], "precision": precision[best], "recall": recall[best]})
threshold_results = pd.DataFrame(threshold_rows).sort_values(["meets_precision_floor", "recall"], ascending=[False, False])
display(threshold_results.round(4))
if not threshold_results["meets_precision_floor"].any():
    print("No candidate reaches 0.50 validation precision while predicting any subscribers. No threshold satisfies the requested objective.")


,model,meets_precision_floor,threshold,precision,recall
7,calibrated_random_forest,False,0.0460,0.1291,0.9594
2,decision_tree,False,0.5838,0.1467,0.8663
0,logistic_regression,False,0.0000,0.1677,0.7411
1,class_weighted_logistic,False,0.0026,0.1675,0.6766
4,random_forest_unweighted,False,0.0570,0.1826,0.3628
3,random_forest_balanced,False,0.5029,0.1775,0.3198
6,random_forest_balanced_larger_leaf,False,0.5154,0.1797,0.2554
5,random_forest_balanced_shallower,False,0.5366,0.1823,0.1181


No candidate reaches 0.50 validation precision while predicting any subscribers. No threshold satisfies the requested objective.


## 9. Candidate Selection for Final Diagnostic Evaluation

No candidate meets the requested 0.50 validation precision floor. For a transparent diagnostic only, select the highest Average Precision candidate. This is not a business-ready selection, and the default 0.50 class threshold remains provisional.


In [14]:
selected_model_name = validation_results.iloc[0]["model"]
selected_threshold = 0.50
print("Top Average Precision candidate:", selected_model_name)
print("Validation Average Precision:", f"{validation_results.iloc[0]['average_precision']:.4f}")
print("Validation precision floor met:", bool(threshold_results.loc[threshold_results.model == selected_model_name, "meets_precision_floor"].iloc[0]))
print("Provisional class threshold:", selected_threshold)


Top Average Precision candidate: random_forest_balanced
Validation Average Precision: 0.1572
Validation precision floor met: False
Provisional class threshold: 0.5


## 10. Final Chronological Segment Evaluation

The model and provisional threshold are fixed above before this evaluation. The final segment was already inspected and scored in earlier notebook versions, so this run is explicitly **post-exposure** and cannot be presented as an untouched final estimate. No model selection follows this cell.


In [15]:
final_metrics, model_artifact_path = run_final_evaluation(
    data_path=data_path,
    model_name=selected_model_name,
    threshold=selected_threshold,
    output_dir=project_root,
    test_status="previously_exposed",
)
print("Selected candidate:", selected_model_name)
print("Test status:", final_metrics["test_status"])
print("Threshold:", final_metrics["threshold"])
print("ROC-AUC:", f"{final_metrics['roc_auc']:.4f}")
print("Average Precision:", f"{final_metrics['average_precision']:.4f}")
print("Precision:", f"{final_metrics['precision']:.4f}")
print("Recall:", f"{final_metrics['recall']:.4f}")
print("F1:", f"{final_metrics['f1']:.4f}")
print("Actual subscriber rate:", f"{final_metrics['actual_subscriber_rate']:.2%}")
print("Predicted subscriber rate:", f"{final_metrics['predicted_subscriber_rate']:.2%}")
print("\nConfusion matrix [no, yes]:")
print(np.asarray(final_metrics["confusion_matrix"]))
print("\nClassification report:")
display(pd.DataFrame(final_metrics["classification_report"]).T.round(4))
print("\nModel artifact:", model_artifact_path)


Selected candidate: random_forest_balanced
Test status: previously_exposed
Threshold: 0.5
ROC-AUC: 0.6959
Average Precision: 0.4840
Precision: 0.3514
Recall: 0.9268
F1: 0.5096
Actual subscriber rate: 30.83%
Predicted subscriber rate: 81.32%

Confusion matrix [no, yes]:
[[1353 4345]
 [ 186 2354]]

Classification report:


,precision,recall,f1-score,support
no,0.8791,0.2375,0.3739,5698.00
yes,0.3514,0.9268,0.5096,2540.00
accuracy,0.4500,0.4500,0.4500,0.45
macro avg,0.6153,0.5821,0.4417,8238.00
weighted avg,0.7164,0.4500,0.4157,8238.00



Model artifact: D:\code\Bank Marketing Response Prediction\bank-marketing-ml\models\selected_model.joblib


## 11. Model Interpretation

The selected tree model?s impurity-based feature importance is descriptive of this fitted model. It does not establish causation or business impact.


In [16]:
importance_path = project_root / "reports" / "feature_importance.csv"
feature_importance = pd.read_csv(importance_path)
display(feature_importance.head(15))


,feature,importance,interpretation
0,num__euribor3m,0.142769,tree_importance
1,num__cons.conf.idx,0.140082,tree_importance
2,num__cons.price.idx,0.100097,tree_importance
3,num__emp.var.rate,0.090030,tree_importance
4,num__nr.employed,0.070003,tree_importance
5,cat__month_apr,0.049215,tree_importance
6,num__age,0.045590,tree_importance
7,cat__contact_cellular,0.039638,tree_importance
8,cat__month_may,0.031120,tree_importance
9,cat__contact_telephone,0.025231,tree_importance
